Load and check the precinct data, boundaries, municipalities, and population totals from the extracted Redistricting folder.


In [1]:
import ast
import math
import re
from pathlib import Path

import pandas as pd
import shapefile as shp
from shapely.geometry import shape

data_folder = Path("Redistricting")
map_folder = data_folder / "Map Data"


In [2]:
nj_precinct_data = pd.read_csv(
    map_folder / "precinct-data-congress-nj.csv",
    dtype={"GEOID20": str}
)

nj_contiguity = pd.read_csv(
    data_folder / "Contiguity_nj.csv",
    header=None,
    names=["Precinct", "Neighbors"],
    dtype=str
)
nj_contiguity["Neighbors"] = nj_contiguity["Neighbors"].apply(ast.literal_eval)

nj_precinct_boundaries = {}
boundary_records = []
with shp.Reader(str(map_folder / "nj_vtd_2020_bound" / "nj_vtd_2020_bound")) as reader:
    for entry in reader.iterShapeRecords():
        record = entry.record.as_dict()
        precinct = record["GEOID20"]
        if precinct in nj_precinct_boundaries:
            raise ValueError(f"Duplicate boundary ID: {precinct}")
        nj_precinct_boundaries[precinct] = shape(entry.shape)
        boundary_records.append(record)

boundary_data = pd.DataFrame(boundary_records)


In [3]:
for column in ["GEOID20", "Total_2020_Total"]:
    if column not in nj_precinct_data.columns:
        raise ValueError(f"Missing data column: {column}")

if len(nj_precinct_data) != 6361:
    raise ValueError(f"Expected 6361 precincts, found {len(nj_precinct_data)}")

if nj_precinct_data["GEOID20"].isna().any() or nj_precinct_data["GEOID20"].duplicated().any():
    raise ValueError("Precinct data has missing or duplicate IDs")

if nj_contiguity["Precinct"].isna().any() or nj_contiguity["Precinct"].duplicated().any():
    raise ValueError("Neighbor data has missing or duplicate IDs")

precinct_ids = set(nj_precinct_data["GEOID20"])
if set(nj_precinct_boundaries) != precinct_ids:
    raise ValueError("Boundary IDs do not match precinct data")
if set(nj_contiguity["Precinct"]) != precinct_ids:
    raise ValueError("Neighbor-index IDs do not match precinct data")

for neighbors in nj_contiguity["Neighbors"]:
    if not isinstance(neighbors, list) or any(not isinstance(p, str) for p in neighbors):
        raise ValueError("Neighbor entries must be lists of precinct ID strings")
    if not set(neighbors).issubset(precinct_ids):
        raise ValueError("Neighbor data references an unknown precinct")

population = pd.to_numeric(nj_precinct_data["Total_2020_Total"], errors="raise")
if population.isna().any() or (population < 0).any() or (population % 1 != 0).any():
    raise ValueError("Populations must be nonnegative whole numbers with no missing values")
nj_precinct_data["Total_2020_Total"] = population.astype("int64")

if any(geom.is_empty for geom in nj_precinct_boundaries.values()):
    raise ValueError("A precinct boundary is empty")

print("Data checks passed: 6361 unique precincts with matching boundaries and neighbor IDs.")


Data checks passed: 6361 unique precincts with matching boundaries and neighbor IDs.


In [4]:
munic_names = {}
munic_groups = {}
unmatched_names = []

for record in boundary_records:
    precinct = record["GEOID20"]
    match = re.match(
        r"^(.+?\b(?:township|borough|city|town|village))\b",
        record["NAME20"],
        flags=re.IGNORECASE
    )
    if match is None:
        munic_names[precinct] = None
        unmatched_names.append({"GEOID20": precinct, "NAME20": record["NAME20"]})
        continue

    name = " ".join(match.group(1).lower().split())
    munic_names[precinct] = name
    key = (record["COUNTYFP20"], name)
    munic_groups.setdefault(key, []).append(precinct)

unmatched_names = pd.DataFrame(unmatched_names, columns=["GEOID20", "NAME20"])
print(f"Municipality groups identified: {len(munic_groups)}")
print(f"Precinct names requiring review: {len(unmatched_names)}")
if not unmatched_names.empty:
    print(unmatched_names.to_string(index=False))
    print("These precincts remain in the data, but are not yet included in municipality groups.")


Municipality groups identified: 565
Precinct names requiring review: 5
    GEOID20                       NAME20
34011ZZZZZZ Voting Districts Not Defined
34009ZZZZZZ Voting Districts Not Defined
34029ZZZZZZ Voting Districts Not Defined
34025ZZZZZZ Voting Districts Not Defined
34001ZZZZZZ Voting Districts Not Defined
These precincts remain in the data, but are not yet included in municipality groups.


In [5]:
target_population = nj_precinct_data["Total_2020_Total"].sum() / 12
min_population = math.ceil(target_population * 0.95)
max_population = math.floor(target_population * 1.05)

print(f"Target population: {target_population:,.2f}")
print(f"Allowed district population: {min_population:,} to {max_population:,}")


Target population: 774,082.83
Allowed district population: 735,379 to 812,786


Build the precinct neighbor graph and look up precinct and municipality populations.


In [6]:
import networkx as nx

graph = nx.Graph()
graph.add_nodes_from(nj_precinct_data["GEOID20"])

for precinct, neighbors in nj_contiguity.itertuples(index=False, name=None):
    for neighbor in neighbors:
        graph.add_edge(precinct, neighbor)

prec_population = nj_precinct_data.set_index("GEOID20")["Total_2020_Total"].to_dict()

muic_population = {}
for munic, precincts in munic_groups.items():
    muic_population[munic] = sum(prec_population[p] for p in precincts)

undef_prec = sorted(unmatched_names["GEOID20"].tolist())


In [7]:
if len(undef_prec) != 5:
    raise ValueError("Expected five undefined precincts; review the unmatched names")
if any(prec_population[p] != 0 for p in undef_prec):
    raise ValueError("An undefined precinct has population; review its handling")

if not nx.is_connected(graph):
    raise ValueError("The full precinct graph is disconnected")

if not nx.is_connected(graph.subgraph(set(graph) - set(undef_prec))):
    raise ValueError("Removing undefined precincts disconnects the graph; review before growth")

for precinct in undef_prec:
    if not any(neighbor not in undef_prec for neighbor in graph.neighbors(precinct)):
        raise ValueError(f"Undefined precinct has no defined neighbor: {precinct}")

if sum(muic_population.values()) != sum(prec_population.values()):
    raise ValueError("Municipality populations do not account for the full population")

print(f"Graph: {graph.number_of_nodes():,} precincts and {graph.number_of_edges():,} neighbor connections")
print("The full graph is connected.")
print("The graph remains connected without the five undefined precincts.")
print(f"Municipality population lookups: {len(muic_population)}")
print(f"Total population: {sum(prec_population.values()):,}")
print("Undefined precincts (all zero population):", undef_prec)


Graph: 6,361 precincts and 17,869 neighbor connections
The full graph is connected.
The graph remains connected without the five undefined precincts.
Municipality population lookups: 565
Total population: 9,288,994
Undefined precincts (all zero population): ['34001ZZZZZZ', '34009ZZZZZZ', '34011ZZZZZZ', '34025ZZZZZZ', '34029ZZZZZZ']


Use the starter functions to check contiguity, calculate district populations, and measure compactness.


In [8]:
import numpy as np
import shapely.ops
from shapely.geometry import Polygon, MultiPolygon

def isDistrictContiguous(district_num, assignment, contiguity_list, print_isolates=False, ignore_list=[]):
    contiguity_list.columns = ['Precinct','Neighbors']
    district_graph = nx.Graph()
    district_nodes = assignment[assignment['District']==district_num]['GEOID20'].tolist()
    for i in ignore_list:
        try:
            district_nodes.remove(i)
        except ValueError:
            pass
    district_graph.add_nodes_from(district_nodes)
    for id in district_nodes:
        neighbors = contiguity_list[contiguity_list['Precinct']==id]['Neighbors'].values.tolist()[0]
        if isinstance(neighbors, str):
            neighbors = ast.literal_eval(neighbors)
        for neighbor in neighbors:
            if neighbor in district_nodes:
                district_graph.add_edge(id,neighbor)
    if(print_isolates):
        print(list(nx.isolates(district_graph)))
    return nx.is_connected(district_graph)

def getDistrictPopulations(assignment,data_file, num_district):
    population = {}
    for i in range (1,num_district+1):
        population[i] = data_file[data_file['GEOID20'].isin(assignment[assignment['District']==i]['GEOID20'])]['Total_2020_Total'].sum()
    return population

def getDistrictShape(district_id, assignment, boundaries):
    list_precincts = assignment[assignment['District']==district_id]['GEOID20']
    precinct_shapes = []
    for i in list_precincts:
        if shape(boundaries[i]).geom_type == 'Polygon':
            precinct_shapes.append(Polygon(shape(boundaries[i])))
        elif shape(boundaries[i]).geom_type == 'MultiPolygon':
            precinct_shapes.append(MultiPolygon(shape(boundaries[i])))      
    district_shape = shapely.ops.unary_union(precinct_shapes)
    return district_shape

def pp_compactness(geom):
    p = geom.length
    a = geom.area    
    return (4*np.pi*a)/(p*p)

def box_reock_compactness(geom):
    a = geom.area 
    bb = geom.bounds
    bba = abs(bb[0]-bb[2])*abs(bb[1]-bb[3])
    return a/bba

Split connected precinct regions into population-balanced districts while preferring fewer municipality splits.


In [9]:
rng = np.random.default_rng(20)


In [10]:
def split_region(precincts, num_districts, attempts):
    precincts = set(precincts)
    total = sum(prec_population[p] for p in precincts)
    if not num_districts * min_population <= total <= num_districts * max_population:
        return None
    region_graph = graph.subgraph(precincts)
    if not nx.is_connected(region_graph):
        return None
    if num_districts == 1:
        return [precincts]

    left_count = num_districts // 2
    right_count = num_districts - left_count
    nodes = sorted(precincts)
    edges = sorted((min(a, b), max(a, b)) for a, b in region_graph.edges())
    local_munic = []
    for group in munic_groups.values():
        group = set(group) & precincts
        if group:
            local_munic.append(group)

    for attempt in range(attempts):
        tree = nx.Graph()
        tree.add_nodes_from(nodes)
        for a, b in edges:
            tree.add_edge(a, b, weight=float(rng.random()))
        tree = nx.minimum_spanning_tree(tree)

        order = [nodes[0]]
        parent = {}
        for a, b in nx.dfs_edges(tree, source=nodes[0]):
            parent[b] = a
            order.append(b)

        population = {p: prec_population[p] for p in nodes}
        size = {p: 1 for p in nodes}
        for p in reversed(order[1:]):
            population[parent[p]] += population[p]
            size[parent[p]] += size[p]

        candidates = []
        for index in range(1, len(order)):
            p = order[index]
            left_pop = population[p]
            right_pop = total - left_pop
            left = None
            if left_count * min_population <= left_pop <= left_count * max_population:
                if right_count * min_population <= right_pop <= right_count * max_population:
                    left = set(order[index:index + size[p]])
            if left is None:
                if left_count * min_population <= right_pop <= left_count * max_population:
                    if right_count * min_population <= left_pop <= right_count * max_population:
                        left = precincts - set(order[index:index + size[p]])
                        left_pop = right_pop
            if left is None:
                continue

            right = precincts - left
            splits = sum(1 for group in local_munic if group & left and group & right)
            difference = abs(left_pop - total * left_count / num_districts)
            candidates.append((splits, difference, left))

        candidates.sort(key=lambda candidate: (candidate[0], candidate[1]))
        for splits, difference, left in candidates:
            left_regions = split_region(left, left_count, attempts)
            if left_regions is None:
                continue
            right_regions = split_region(precincts - left, right_count, attempts)
            if right_regions is not None:
                return left_regions + right_regions

    return None



Build a complete map and verify all precinct assignments, district populations, and contiguity.


In [11]:
def build_map(attempts):
    if attempts < 1:
        raise ValueError("attempts must be a positive integer")
    regions = split_region(set(graph) - set(undef_prec), 12, attempts)
    if regions is None:
        print("No complete draft found within the tree-attempt limit.")
        return None

    map = nj_precinct_data[["GEOID20"]].copy()
    map["District"] = 0
    for district, district_prec in enumerate(regions, start=1):
        map.loc[map["GEOID20"].isin(district_prec), "District"] = district

    assignment = map.set_index("GEOID20")["District"].to_dict()
    possible_maps = [map]
    for precinct in undef_prec:
        counts = {}
        for neighbor in graph.neighbors(precinct):
            if neighbor in undef_prec:
                continue
            district = assignment[neighbor]
            if district > 0:
                counts[district] = counts.get(district, 0) + 1
        if not counts:
            raise ValueError(f"No assigned neighboring district for {precinct}")
        choices = sorted(district for district in counts if counts[district] == max(counts.values()))
        updated_maps = []
        for trial in possible_maps:
            for district in choices:
                updated = trial.copy()
                updated.loc[updated["GEOID20"] == precinct, "District"] = district
                updated_maps.append(updated)
        possible_maps = updated_maps

    map = possible_maps[0]
    if len(possible_maps) > 1:
        best_compactness = None
        for trial in possible_maps:
            compactness = 0
            for district in range(1, 13):
                district_shape = getDistrictShape(district, trial, nj_precinct_boundaries)
                compactness += pp_compactness(district_shape)
            compactness /= 12
            if best_compactness is None or compactness > best_compactness:
                map = trial
                best_compactness = compactness

    if len(map) != 6361 or map["GEOID20"].duplicated().any():
        raise ValueError("Expected exactly 6361 unique precinct assignments")
    if set(map["GEOID20"]) != set(graph):
        raise ValueError("Missing or unexpected precinct IDs in the map")
    if set(map["District"]) != set(range(1, 13)):
        raise ValueError("Every precinct must be assigned to one of districts 1 through 12")

    population = getDistrictPopulations(map, nj_precinct_data, 12)
    for district in range(1, 13):
        if not min_population <= population[district] <= max_population:
            raise ValueError(f"District {district} is outside the population limits")
        if not isDistrictContiguous(district, map, nj_contiguity):
            raise ValueError(f"District {district} is not contiguous")

    print("All 6361 precincts assigned exactly once to 12 contiguous districts.")
    print(f"District populations: {min(population.values()):,} to {max(population.values()):,}; all within ±5%.")
    return map


Generate 10 valid maps from seed 20 and choose the lowest municipality split penalty, using average compactness to break ties.


In [12]:
candidate_count = 10
tree_attempts = 50
rng = np.random.default_rng(20)

best_map = None
best_splits = None
best_compactness = None
best_candidate = None
candidate_results = []

for candidate in range(1, candidate_count + 1):
    map = build_map(tree_attempts)
    if map is None:
        raise ValueError(f"Candidate {candidate} could not be built; review the retry limit")

    assignment = map.set_index("GEOID20")["District"]
    split_penalty = 0
    for precincts in munic_groups.values():
        split_penalty += assignment.loc[precincts].nunique() - 1

    compactness = 0
    for district in range(1, 13):
        district_shape = getDistrictShape(district, map, nj_precinct_boundaries)
        compactness += pp_compactness(district_shape)
    avg_compactness = compactness / 12
    if not math.isfinite(avg_compactness):
        raise ValueError(f"Candidate {candidate} has an invalid compactness score")

    candidate_results.append({
        "Candidate": candidate,
        "Extra municipality pieces": int(split_penalty),
        "Average Polsby-Popper": float(avg_compactness)
    })

    if best_map is None or split_penalty < best_splits or (split_penalty == best_splits and avg_compactness > best_compactness):
        best_map = map.copy()
        best_splits = int(split_penalty)
        best_compactness = float(avg_compactness)
        best_candidate = candidate

    print(f"Candidate {candidate}: split penalty {split_penalty}, average compactness {avg_compactness:.4f}")

candidate_results = pd.DataFrame(candidate_results)
print(f"Selected candidate {best_candidate}: split penalty {best_splits}, average compactness {best_compactness:.4f}")
candidate_results


All 6361 precincts assigned exactly once to 12 contiguous districts.
District populations: 735,906 to 811,024; all within ±5%.
Candidate 1: split penalty 131, average compactness 0.1742
All 6361 precincts assigned exactly once to 12 contiguous districts.
District populations: 740,164 to 809,285; all within ±5%.
Candidate 2: split penalty 108, average compactness 0.1671
All 6361 precincts assigned exactly once to 12 contiguous districts.
District populations: 735,819 to 810,085; all within ±5%.
Candidate 3: split penalty 116, average compactness 0.1824
All 6361 precincts assigned exactly once to 12 contiguous districts.
District populations: 735,409 to 812,257; all within ±5%.
Candidate 4: split penalty 108, average compactness 0.1879
All 6361 precincts assigned exactly once to 12 contiguous districts.
District populations: 738,530 to 812,205; all within ±5%.
Candidate 5: split penalty 105, average compactness 0.2073
All 6361 precincts assigned exactly once to 12 contiguous districts.
D

Candidate,Extra municipality pieces,Average Polsby-Popper
1,131,0.174154
2,108,0.167056
3,116,0.182374
4,108,0.187915
5,105,0.207277
6,93,0.218852
7,105,0.179190
8,107,0.196271
9,122,0.169699
10,108,0.183954


Save the selected map as fair_map_gp.csv and check that the exported assignments match.


In [13]:
if best_map is None or len(candidate_results) != candidate_count:
    raise ValueError("Complete the candidate comparison before exporting")

output_file = "fair_map_gp.csv"
best_map[["GEOID20", "District"]].to_csv(output_file, index=False)

exported = pd.read_csv(output_file, dtype={"GEOID20": str})
if not exported.equals(best_map[["GEOID20", "District"]].reset_index(drop=True)):
    raise ValueError("Exported assignments do not match the selected map")
print(f"Saved {output_file} with {len(exported):,} precinct assignments.")


Saved fair_map_gp.csv with 6,361 precinct assignments.
